<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Backend Module 1 (a): Python & OOP Refresher

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Refresh the **Python a backend actually uses** — collections, functions, comprehensions
2. Write **type hints**, and see why FastAPI cares about them more than Python does
3. Handle and **raise** errors, and move data in and out of **JSON**
4. Build **classes and objects**, and meet `self` properly
5. Work through the **four pillars** of OOP, one small example each
6. Use **dunders**, `@property`, **composition** and `@dataclass`
7. Finish with **Pydantic** — a class that checks itself, which is what FastAPI runs on

> **Nothing to install for sections 2–13** — this is plain Python. Section 14 installs Pydantic
> in one line. No API keys anywhere.
>
> **Run it in Colab while your laptop sets itself up.** The FastAPI half of this module runs
> locally in VS Code, and `uv sync` takes a few minutes — start that download, then come back
> here and start typing.

---

## 1. How to Use This Notebook

Run every cell top to bottom, in order — later cells use names defined earlier.

**Don't just read it.** After a cell runs, change a value and run it again. That habit is worth more
than any explanation here.

The three parts:

| Part | Sections | What it is |
|---|---|---|
| **A · Python** | 2–8 | the parts of Python a backend leans on all day |
| **B · OOP** | 9–13 | classes, the four pillars, and the shortcuts |
| **C · Pydantic** | 14–16 | a class that validates itself — FastAPI's engine |

---

# Part A — The Python You Actually Need

---

## 2. Variables and the Four Collections

In [ ]:
name = "Ada"
age = 20

print(f"{name} is {age} years old")      # f-string: put a value straight into text

Four ways to hold several things. Choosing the right one is most of the skill.

In [ ]:
marks = [90, 85, 77]                        # list  - ordered, changeable
point = (12, 5)                             # tuple - ordered, FIXED
student = {"name": "Ada", "branch": "CSE"}  # dict  - labelled data   <- the big one
branches = {"CSE", "ECE", "CSE"}            # set   - no duplicates

print(marks[0], point[0], student["name"], branches)

> **A dict is how data travels in a backend.** Every API request and every API response is,
> underneath, a dict. Remember this shape — you will see it all week.

In [ ]:
print(student["branch"])                 # square brackets: crashes if the key is missing
print(student.get("age", "not given"))   # .get(): returns a fallback instead of crashing

---

## 3. Deciding and Repeating

Nothing surprising here — but note the second loop. Looping over a dict gives you the **keys**,
and `.items()` gives you both.

In [ ]:
if age >= 18:
    print("adult")
else:
    print("minor")

for mark in marks:
    print("mark:", mark)

for key, value in student.items():
    print(key, "->", value)

total = 0
while total < 100:
    total += 50
print("total:", total)

---

## 4. Functions

A function is a named block you can run again. Two things worth naming: **default arguments**, and
**keyword arguments**.

In [ ]:
def average(numbers: list[int]) -> float:      # the : and -> are TYPE HINTS (section 6)
    return sum(numbers) / len(numbers)


def greet(name: str, greeting: str = "Hi") -> str:    # greeting has a DEFAULT
    return f"{greeting} {name}"


print(average(marks))
print(greet("Ada"))                          # uses the default
print(greet("Ada", greeting="Welcome"))      # keyword argument - clearer to read

---

## 5. Comprehensions, and Three Gotchas

A comprehension builds a list in one line. You will read a lot of these.

In [ ]:
squares = [n * n for n in marks]
toppers = [m for m in marks if m > 80]       # with a filter

print(squares, toppers)

### Three things that catch people out

In [ ]:
print(4 / 2)        # 2.0   <- division ALWAYS gives a float
print(4 // 2)       # 2      <- use // when you want a whole number

a = [1, 2]
b = [1, 2]
print(a == b)       # True   <- same contents
print(a is b)       # False  <- not the same object in memory


def add_item(item, basket=None):
    # DON'T write basket=[] - Python creates that list ONCE and reuses it forever,
    # so every call would share the same basket.
    if basket is None:
        basket = []
    basket.append(item)
    return basket


print(add_item("pen"), add_item("book"))     # ['pen'] ['book'] - fresh each time

---

## 6. Type Hints — Why a Backend Cares

A hint says what a value is **supposed** to be.

In [ ]:
def double(n: int) -> int:
    return n * 2


print(double(5))

# Python does NOT enforce hints. This runs, and gives nonsense:
print(double("ab"))        # abab   <- no error, no complaint

So why bother?

1. Your editor autocompletes and warns you.
2. Other people read them as documentation.
3. **FastAPI reads them and turns them into real validation.**

That third one is why this course cares. The same hint, enforced for free.

In [ ]:
def totals(marks: list[int]) -> int:
    return sum(marks)


def label(student: dict[str, str]) -> str:
    return f"{student['name']} ({student['branch']})"


def find(name: str) -> str | None:           # "a string, OR nothing at all"
    return None if name == "" else name


print(totals([90, 85]))
print(label({"name": "Ada", "branch": "CSE"}))
print(find(""), "|", find("Ada"))

> **Where this is going.** Later in this module you will write:
>
> ```python
> @app.get("/students/{student_id}")
> def get_student(student_id: int):
>     ...
> ```
>
> and `"abc"` will be rejected with a clear error, automatically — because of that one word: `int`.

---

## 7. Errors — try / except / raise

A backend spends its day checking input and refusing it clearly.

In [ ]:
try:
    age = int("hello")
except ValueError:
    print("that was not a number")

`raise` is how **you** refuse something on purpose.

In [ ]:
def set_age(age: int) -> int:
    if age < 0:
        raise ValueError("age cannot be negative")
    return age


try:
    set_age(-5)
except ValueError as error:
    print("refused:", error)

# This is exactly what a backend does all day: check, and refuse clearly.
# In FastAPI the same idea has an API-shaped version:  raise HTTPException(404)

`finally` runs either way — whether it worked or blew up.

In [ ]:
try:
    number = int("42")
except ValueError:
    number = 0
finally:
    print("checked the input, moving on ->", number)

---

## 8. JSON — a Dict With Different Punctuation

JSON is the format every API on the internet speaks. It looks almost exactly like a Python dict.

In [ ]:
import json

student = {"name": "Ada", "branch": "CSE", "age": 20, "active": True}

as_text = json.dumps(student)          # Python dict -> JSON text
print(as_text)
print(type(as_text))                   # <class 'str'>  <- it is TEXT now

back_again = json.loads(as_text)       # JSON text -> Python dict
print(back_again["name"], type(back_again))

**Spot the two differences** in the printed JSON: `True` became `true`, and the single quotes
became double quotes. Those are the only real changes.

> Good news: FastAPI does both conversions for you. You return a dict; it sends JSON.

---

# Part B — OOP

---

## Why a backend cares about OOP

You already have dicts and functions. So why classes?

Because a dict has **no rules**. Nothing stops `{"age": -5}`, or a typo turning `branch` into
`brnach`, or one part of your code setting a field another part assumed could never change. The data
and the rules that protect it live in different places — and the rules only run if whoever wrote the
calling code remembered to run them.

**A class puts the data and its rules in the same box.** An object can refuse to exist in a broken
state, and then every other piece of code can stop checking.

You will meet this three times before the module ends:

| Where | What it is |
|---|---|
| **Pydantic models** | a class that validates every field for you *(section 14)* |
| **FastAPI routers** | classes and decorators grouping related endpoints |
| **Database models** | one class per table, one object per row |

None of that makes sense without the next five sections.

---

## 9. Classes and Objects

A **class** is a blueprint. An **object** is one thing built from it.

```
   class Student      ->   the form:  every student has an id, a name, a branch
   Student(1, "Ada")  ->   one filled-in copy of that form
   Student(2, "Raj")  ->   another, completely separate
```

One class, as many objects as you like — each with its own data.

In [ ]:
class Student:
    college = "LPU"           # CLASS attribute - shared by every student

    def __init__(self, id: int, name: str, branch: str = "CSE"):
        # INSTANCE attributes - each object gets its own copy
        self.id = id
        self.name = name
        self.branch = branch

    def label(self) -> str:                  # a method = a function inside a class
        return f"[{self.id}] {self.name} ({self.branch})"


# One blueprint -> as many objects as you want
ada = Student(1, "Ada")
raj = Student(2, "Raj", "ECE")

print(ada.label())
print(raj.label())

### `__init__` — the setup method

`__init__` runs **automatically**, once, the moment you write `Student(1, "Ada")`. You never call it
yourself. Its job is to take the arguments and hang them on the new object as attributes.

Other languages call this a **constructor**. Python calls it "the init method", and it is the first
of many **dunders** — you will meet the rest in section 11.

### What is `self`, really?

`self` is just **"the object this method was called on"**. When you write `ada.label()`, Python
quietly passes `ada` in as `self`. Here is the proof — both lines do the same thing.

In [ ]:
print(ada.label())
print(Student.label(ada))      # exactly the same output

Java and C++ hide this and give you an invisible `this`. Python's rule is *explicit is better
than implicit*: if a method receives the object, you can see it receiving the object.

Two consequences worth remembering:

- **Every method needs `self` as its first parameter.** Forgetting it is the single most common
  beginner error, and the message (`takes 0 positional arguments but 1 was given`) never mentions it.
- **`self` is a convention, not a keyword.** You *could* call it `banana`. Don't.

### Instance vs class attributes

An **instance** attribute belongs to one object. A **class** attribute is shared by all of them —
useful for something genuinely common to every instance, like a constant or a counter.

In [ ]:
print(ada.college, raj.college)        # LPU LPU - both read the same shared value

ada.name = "Ada Lovelace"              # changing an INSTANCE attribute...
print(ada.name, "|", raj.name)         # ...only affects ada

Student.college = "LPU Jalandhar"      # changing the CLASS attribute...
print(ada.college, "|", raj.college)   # ...affects everyone

⚠️ **Never make a class attribute a list or dict** — every object would share the *same* list, and
changing it through one object changes it for all of them. Exactly the mutable-default trap from
section 5, wearing a different hat. Set those in `__init__` instead.

---

## 10. The Four Pillars

Four ideas, one small example each. They are not four separate techniques — they are four angles on
the same goal: **code you can change safely later**.

### 1) Encapsulation — keep the data and the rule that protects it together

The point is not secrecy. The point is that **an object should never be able to end up in an invalid
state.**

If the balance can only change through `deposit()`, and `deposit()` refuses negative amounts, then
"balance is never negative" is guaranteed by the class itself. Nobody calling it has to remember —
and *"the caller has to remember"* is where bugs come from.

In [ ]:
class BankAccount:
    def __init__(self):
        self._balance = 0                     # the _ means "internal, hands off"

    def deposit(self, amount: int):
        if amount <= 0:
            raise ValueError("amount must be positive")   # the rule lives HERE
        self._balance += amount

    def balance(self) -> int:
        return self._balance


account = BankAccount()
account.deposit(500)
print("Balance:", account.balance())

try:
    account.deposit(-100)
except ValueError as error:
    print("refused:", error)

> **A single underscore is a convention, not a lock.** `account._balance = -999` still works —
> Python has no `private` keyword and does not stop you. It is a sign that reads *"this is internal,
> I may change it, don't build on it."* Python trusts you and documents intent instead of enforcing
> it. Respecting that sign is what separates a library you can upgrade from one you can't.

### 2) Inheritance — a child class reuses its parent

`Admin` gets everything `User` has, for free, and then changes only what differs. Write the common
part once.

In [ ]:
class User:
    def __init__(self, name: str):
        self.name = name

    def role(self) -> str:
        return "user"

    def describe(self) -> str:
        return f"{self.name} -> role: {self.role()}"


class Admin(User):
    def __init__(self, name: str, level: int):
        super().__init__(name)         # super() = "run the parent's version first"
        self.level = level             # then add what is new

    def role(self) -> str:             # same method name, different answer
        return "admin"


print(User("Ada").describe())
print(Admin("Raj", 2).describe())      # describe() came from User, role() from Admin

Three things in that example:

- **`super().__init__(name)`** runs the parent's setup, so you don't repeat `self.name = name`. Do
  this *before* adding the child's own attributes.
- **Overriding**: `Admin.role()` replaces `User.role()`. Same name, different answer.
- **The interesting one** — `describe()` was written once, in `User`, and never touched again. Yet it
  prints "admin" for an Admin, because `self.role()` is looked up on **the actual object** at the
  moment it runs. That is polymorphism, already at work. Hold that thought.

⚠️ **Use inheritance sparingly.** It is the most overused of the four pillars, because it is the one
that feels cleverest. The test is the phrase **"is-a"**: an Admin *is a* User, so this is fine. When
the honest phrase is "has-a", inheritance is the wrong tool — section 12.

### 3) Polymorphism — same message, different behaviour

Two unrelated classes, both with a `send()` method. Neither inherits from anything.

In [ ]:
class EmailNotifier:
    def send(self, message: str) -> str:
        return f"EMAIL: {message}"


class SMSNotifier:
    def send(self, message: str) -> str:
        return f"SMS: {message}"


notifier = EmailNotifier()      # <- change this to SMSNotifier() and run again
print(notifier.send("Your marks are out"))

Swapping that one line proves the two are interchangeable — but *you* did the swapping, by editing
the file. The real payoff is code that does it at **run time**, without knowing or caring which class
it was handed.

In [ ]:
def broadcast(notifiers: list, message: str):
    """This function never mentions EmailNotifier or SMSNotifier."""
    for notifier in notifiers:
        print(notifier.send(message))        # whatever it is, it can send()


broadcast([EmailNotifier(), SMSNotifier()], "Your marks are out")

**That is polymorphism.** `broadcast` is written once, names no class, and would handle a
`WhatsAppNotifier` you write tomorrow without a single change.

> **Duck typing** — Python's version of the idea. *"If it walks like a duck and quacks like a duck,
> it's a duck."* Python does not check that both classes share a base class or an interface. It just
> calls `.send()` and sees whether the object has one. **Polymorphism in Python does not require
> inheritance** — that surprises people arriving from Java or C++.

Notice you have now seen it twice: `describe()` in the inheritance example calling the right
`role()`, and `broadcast()` here calling the right `send()`. One works through a parent class, one
through duck typing. Both are *"the calling code doesn't change when the object does."*

### 4) Abstraction — show WHAT it does, hide HOW it does it

In [ ]:
class Database:
    def save(self, student: str) -> str:
        row = self._format(student)
        return self._write_to_disk(row)       # the caller sees none of this

    def _format(self, student: str) -> str:
        return student.strip().title()

    def _write_to_disk(self, row: str) -> str:
        return f"saved {row}"


print(Database().save("  ada  "))             # you call save(). That is all you need to know.

> ⚠️ **Abstraction vs encapsulation — the pair everyone confuses.** Both use a `_` and both involve
> the word "hide", so here is the difference in one line each:
>
> - **Encapsulation** protects **data** — it stops the balance going negative.
> - **Abstraction** hides **complexity** — it means you call `save()` without knowing there are two
>   steps behind it.
>
> You can have either without the other. `save()` hides complexity but guards no data. A class could
> guard its data through one setter and still expose a messy interface. They travel together in
> practice, which is exactly why they get muddled.

You use abstraction constantly without noticing: `json.dumps()`, `print()`, and every FastAPI
decorator you are about to write. You know what they do. You have no idea how.

### The four pillars, in one table

| Pillar | In one line | In code | Where you'll meet it |
|---|---|---|---|
| **Encapsulation** | data and its rules in one box | `_balance` + `deposit()` | Pydantic refusing bad input |
| **Inheritance** | a child reuses its parent | `class Admin(User)` | your own exception classes |
| **Polymorphism** | same call, different behaviour | `notifier.send(...)` | swapping one database for another |
| **Abstraction** | what, not how | `save()` hiding two steps | every framework you will ever use |

---

## 11. Dunders and `@property`

First, what you get **without** them.

In [ ]:
class Plain:
    def __init__(self, name: str):
        self.name = name


print(Plain("Ada"))                    # <__main__.Plain object at 0x...>  - useless
print(Plain("Ada") == Plain("Ada"))    # False - two different objects in memory

### What a dunder is

**Dunder** = **d**ouble **under**score, as in `__init__`. These are methods **Python calls for you**
when something happens to your object:

| You write | Python calls |
|---|---|
| `print(obj)` | `obj.__str__()` |
| `obj` in a list, or in the debugger | `obj.__repr__()` |
| `a == b` | `a.__eq__(b)` |
| `len(obj)` | `obj.__len__()` |
| `obj[0]` | `obj.__getitem__(0)` |

So `print` isn't doing anything clever — it is asking your object how it would like to be shown, and
the default answer is that useless memory address. Define the dunder and you change the answer.

> This is why `+` works on both numbers and strings: `int` and `str` each define `__add__`
> differently. **Operator overloading is just polymorphism through dunders.**

In [ ]:
class Student:
    def __init__(self, id: int, name: str):
        self.id = id
        self.name = name

    def __str__(self) -> str:              # what print() shows
        return f"{self.name} (#{self.id})"

    def __repr__(self) -> str:             # what the debugger / a list shows
        return f"Student(id={self.id}, name={self.name!r})"

    def __eq__(self, other) -> bool:       # what == means for this class
        return self.id == other.id


ada = Student(1, "Ada")
print(ada)                                 # Ada (#1)
print([ada])                               # [Student(id=1, name='Ada')]   <- __repr__
print(ada == Student(1, "Ada"))            # True - same id, so "the same student"

### `__str__` vs `__repr__` — the actual rule

Both turn your object into text. The difference is **who is reading**:

- **`__str__`** is for a **user**. Readable, friendly. → `Ada (#1)`
- **`__repr__`** is for a **developer**. Unambiguous, ideally something you could paste back into
  Python to rebuild the object. → `Student(id=1, name='Ada')`

Define only one? Make it `__repr__` — Python falls back to it when `__str__` is missing, so you get
both. It is also what shows inside lists, which is where you usually need it most.

And notice `__eq__`: without it, `==` compares **memory addresses**, so two identical students are
"different". With it, *you* decide what "the same student" means — here, same id.

> Keep this in mind for section 13: `@dataclass` writes `__init__`, `__repr__` and `__eq__` for you.
> Now you know exactly what it wrote.

### `@property` — a method that behaves like an attribute

In [ ]:
class Circle:
    def __init__(self, radius: float):
        self.radius = radius

    @property
    def area(self) -> float:               # note: no () when you use it
        return 3.14159 * self.radius ** 2


circle = Circle(2)
print(circle.area)                         # looks like data, is actually code

**Why not just write `area()` as a normal method?** Because of what happens later.

Suppose you shipped `circle.area` as a plain stored attribute, and a hundred places already read it.
Now you need it computed from the radius instead. Without `@property` you would change it to
`area()` and break every one of those hundred callers. With `@property` you swap the implementation
and **nobody changes a line**.

> The rule of thumb: if it *reads* like data — a fact about the object, cheap to work out — expose it
> as a property. If it *does* something, or is slow, make it a method with `()` so the cost is
> visible.

### `@property` + setter — encapsulation, the Python way

Section 10 protected the balance with a `deposit()` method. A **setter** does the same job while
still letting callers write a plain `=`. Same guarantee, more natural syntax.

In [ ]:
class Account:
    def __init__(self):
        self._balance = 0

    @property
    def balance(self) -> int:
        return self._balance

    @balance.setter
    def balance(self, amount: int):
        if amount < 0:
            raise ValueError("balance cannot be negative")   # guarded assignment
        self._balance = amount


account = Account()
account.balance = 500                      # goes through the setter
print(account.balance)

try:
    account.balance = -100                 # ...and the rule still applies
except ValueError as error:
    print("refused:", error)

---

## 12. Composition vs Inheritance

```
  INHERITANCE is "IS-A".     Admin IS A User.
  COMPOSITION is "HAS-A".    Student HAS courses.
```

Ask **"is-a or has-a?"** every time. Most real modelling turns out to be has-a.

In [ ]:
class Course:
    def __init__(self, code: str, title: str):
        self.code = code
        self.title = title

    def __repr__(self) -> str:
        return f"{self.code}"


class Student:
    def __init__(self, name: str):
        self.name = name
        self.courses: list[Course] = []      # HAS-A: a student holds courses

    def enroll(self, course: Course):
        self.courses.append(course)


ada = Student("Ada")
ada.enroll(Course("CS101", "Databases"))
ada.enroll(Course("CS102", "Networks"))
print(ada.name, "->", ada.courses)

# A Student is NOT a Course, so inheritance would be wrong here.

### "Favour composition over inheritance"

You will hear this repeated everywhere. The reasoning:

**Inheritance couples you to a parent you don't control.** A child class can see and depend on its
parent's internals, so a change inside the parent can silently break every child — the *fragile base
class* problem. And a class can only have one natural parent, so as soon as something needs to be
two things at once, the hierarchy starts to bend.

**Composition just holds a reference.** `Student` uses `Course` through its public methods. Change
`Course` internally and `Student` neither knows nor cares. You can swap the held object for a
different one entirely, at run time.

A worked counter-example. Say `Course` has a `title` and a `code`, and you think *"a Classroom is
basically a course with a room number"*:

```python
class Classroom(Course):      # ✗ WRONG - a Classroom is not a kind of Course
    ...
class Classroom:              # ✓ RIGHT
    def __init__(self, room: str, course: Course):
        self.room = room
        self.course = course  # HAS-A
```

Read them aloud. "A classroom is a course" is false. "A classroom has a course" is true. **The
sentence that reads true is the design that will survive.**

> Remember this shape. When you get to databases, that `list[Course]` gets a formal name:
> a **one-to-many relationship**.

### Abstract base classes — a contract children must honour

Sometimes you want to say *"whatever this is, it must be able to `send()`"*. An **ABC** is that
promise, checked by Python.

In [ ]:
from abc import ABC, abstractmethod


class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> str:
        ...                                  # no body - children MUST write it


class EmailNotifier(Notifier):
    def send(self, message: str) -> str:
        return f"EMAIL: {message}"


print(EmailNotifier().send("Marks are out"))


# Forget to write send(), and Python refuses to even create the object:
class BrokenNotifier(Notifier):
    pass


try:
    BrokenNotifier()
except TypeError as error:
    print("refused:", error)

**Why bother, when duck typing already works?** Because of *when* you find out.

With plain duck typing, a missing `send()` blows up whenever some unlucky line finally calls it —
possibly in production, possibly months later. With an ABC it fails **the instant anyone tries to
create the object**, with a message naming the missing method. You have moved a run-time surprise to
the earliest possible moment.

Use an ABC when several classes must be interchangeable and forgetting a method would be a real bug.
Skip it for two throwaway classes — duck typing is enough, and Python programmers reach for it first.

---

## 13. `@dataclass`

Most classes are just "a bag of fields". Writing `__init__`, `__repr__` and `__eq__` by hand for
those gets old fast — and every line of boilerplate is a line that can be wrong, drift out of sync
with the fields, or get half-updated when someone adds a field.

Here is the same class both ways.

In [ ]:
# By hand
class StudentByHand:
    def __init__(self, id: int, name: str, branch: str = "CSE"):
        self.id = id
        self.name = name
        self.branch = branch

    def __repr__(self):
        return f"StudentByHand(id={self.id}, name={self.name!r}, branch={self.branch!r})"

    def __eq__(self, other):
        return (self.id, self.name, self.branch) == (other.id, other.name, other.branch)


print(StudentByHand(1, "Ada"))

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Student:
    id: int
    name: str
    branch: str = "CSE"          # __init__, __repr__ and __eq__ are written for you

    def label(self) -> str:      # your own methods still work normally
        return f"[{self.id}] {self.name} ({self.branch})"


ada = Student(1, "Ada")
print(ada)                              # Student(id=1, name='Ada', branch='CSE')
print(ada.label())
print(ada == Student(1, "Ada"))         # True - compares by value

Fourteen lines became four. Note what the decorator read: **the type hints**. `id: int` is not
decoration here — it is how `@dataclass` knows there is a field called `id` at all. Section 6's
"hints are just documentation" stops being true the moment a library starts reading them, and this
is the first of three times you will see that today.

Useful options: `@dataclass(frozen=True)` makes objects immutable (and hashable, so they work as dict
keys), and `@dataclass(order=True)` adds `<`, `>` and friends so you can sort them.

⚠️ **A list inside a dataclass needs `field(default_factory=list)`** — for the same reason as the
mutable-default gotcha in section 5.

In [ ]:
@dataclass
class Classroom:
    room: str
    students: list[str] = field(default_factory=list)    # NOT  students: list = []


a = Classroom("A1")
b = Classroom("B2")
a.students.append("Ada")

print(a, "|", b)        # b is still empty - each object got its own list

### Where `@dataclass` stops

Try this: `Student(id="abc", name=123, branch=None)`.

It works. No error. `@dataclass` reads your type hints to find the *fields* — it never checks that
the values match. It writes boilerplate; it does not validate.

That is fine for data you created yourself. It is **not** fine for data that arrived over the
internet from someone you have never met, which is every single request your API will ever handle.

> For that you want a class that reads the same hints and **enforces** them. That is Pydantic, and
> it is the next section.

---

# Part C — Pydantic

---

## 14. Pydantic — a Class That Checks Itself

A `@dataclass` holds fields. A **Pydantic model** holds fields *and enforces what they are*.

That difference is the whole reason FastAPI can validate a request without you writing a single
`if`.

In [ ]:
# The only install in this notebook
!pip install -q pydantic email-validator

In [ ]:
from pydantic import BaseModel, ValidationError


class Student(BaseModel):
    name: str                 # required - no default
    age: int                  # required
    branch: str = "CSE"       # optional - has a default
    email: str | None = None  # optional - may genuinely be nothing


ada = Student(name="Ada", age=20)
print(ada)

### Coercion — it converts when it safely can

In [ ]:
raj = Student(name="Raj", age="21")        # a STRING went in...
print(raj.age, type(raj.age))              # 21 <class 'int'>   ...an int came out

### Rejection — when it cannot convert, it refuses

In [ ]:
try:
    Student(name="Meera", age="hello")
except ValidationError as error:
    print(error)

### The error is **data**, not just a message

This is the part that matters for an API. You can loop over the problems and do something with
them — which is exactly what FastAPI does to build a `422` response.

In [ ]:
try:
    Student(age="hello")                   # two problems: name missing, age unparseable
except ValidationError as error:
    for problem in error.errors():
        print(problem["loc"], "->", problem["msg"])

### Getting data back out

In [ ]:
print(ada.model_dump())          # -> a plain dict
print(ada.model_dump_json())     # -> a JSON string

# ...and going the other way, from data you received:
incoming = {"name": "Meera", "age": 19, "branch": "ECE"}
print(Student.model_validate(incoming))

---

## 15. Field Rules and Validators

`Field()` puts rules on a single value. In words:

```
  gt  >   greater than          lt  <   less than
  ge  >=  greater or equal      le  <=  less or equal
```

In [ ]:
from pydantic import Field


class Applicant(BaseModel):
    name: str = Field(
        min_length=2,
        max_length=50,
        description="Full name",          # this shows up in /docs later
        examples=["Ada Lovelace"],
    )
    age: int = Field(ge=16, le=100)
    cgpa: float = Field(gt=0, lt=10)
    roll_no: str = Field(pattern=r"^\d{8}$")   # exactly 8 digits
    branch: str = "CSE"


print(Applicant(name="Ada", age=20, cgpa=9.1, roll_no="12345678"))

In [ ]:
def try_it(model, **values):
    """Build the model and print any problems instead of crashing."""
    try:
        model(**values)
    except ValidationError as error:
        for problem in error.errors():
            # loc says WHICH field failed. It is empty for whole-model rules (section 15).
            where = problem["loc"][0] if problem["loc"] else "(whole model)"
            print(where, "->", problem["msg"])


try_it(Applicant, name="A",   age=20, cgpa=9.1,  roll_no="12345678")   # name too short
try_it(Applicant, name="Ada", age=12, cgpa=9.1,  roll_no="12345678")   # too young
try_it(Applicant, name="Ada", age=20, cgpa=11.0, roll_no="12345678")   # cgpa out of range
try_it(Applicant, name="Ada", age=20, cgpa=9.1,  roll_no="ABC")        # wrong shape

### Validators — rules `Field()` cannot express

A **`field_validator`** checks or cleans one field. A **`model_validator`** runs once the whole
object exists, so it can compare fields against each other.

In [ ]:
from pydantic import field_validator, model_validator


class Signup(BaseModel):
    username: str
    password: str
    confirm_password: str
    branch: str

    @field_validator("username")
    @classmethod
    def username_must_be_lowercase(cls, value: str) -> str:
        if not value.islower():
            raise ValueError("username must be lowercase")
        return value

    @field_validator("branch")
    @classmethod
    def tidy_branch(cls, value: str) -> str:
        return value.strip().upper()          # a validator can CLEAN, not just reject

    @model_validator(mode="after")
    def passwords_must_match(self):
        if self.password != self.confirm_password:
            raise ValueError("passwords do not match")
        return self


print(Signup(username="ada", password="x1", confirm_password="x1", branch="  cse "))

try_it(Signup, username="Ada", password="x1", confirm_password="x1", branch="cse")
try_it(Signup, username="ada", password="x1", confirm_password="x2", branch="cse")

> **Why both exist:** "passwords must match" is a rule about the *whole form*, not about one box.
> A `field_validator` cannot see the other fields. A `model_validator` can.

**Look at the last two lines of output.** The username error names a field; the password error says
`(whole model)`. That is not a quirk — a `model_validator` error has **no field location**, because
it isn't about any single field. Your API has to report those two kinds of error differently: one
can highlight a box in the form, the other can only show a message at the top.

---

## 16. Real Types, Nested Models, and Locking It Down

Some types validate themselves.

In [ ]:
from datetime import date
from enum import Enum
from typing import Literal
from uuid import UUID

from pydantic import EmailStr, HttpUrl


class Branch(str, Enum):            # a fixed set of allowed values
    cse = "CSE"
    ece = "ECE"
    mech = "MECH"


class Profile(BaseModel):
    id: UUID
    email: EmailStr
    website: HttpUrl
    joined: date
    branch: Branch
    status: Literal["active", "inactive"]    # like an Enum, written inline


profile = Profile(
    id="123e4567-e89b-12d3-a456-426614174000",
    email="ada@lpu.in",
    website="https://lpu.in",
    joined="2026-08-01",            # a string goes in, a real date comes out
    branch="CSE",
    status="active",
)

print(profile.joined, type(profile.joined))
print(profile.branch, profile.branch.value)

### Nested models — a model inside a model

In [ ]:
class Address(BaseModel):
    city: str
    pincode: str


class StudentProfile(BaseModel):
    name: str
    address: Address                # validated too, all the way down
    courses: list[str] = []


student = StudentProfile(
    name="Ada",
    address={"city": "Jalandhar", "pincode": "144411"},   # a dict becomes an Address
    courses=["CS101"],
)

print(student.address.city, type(student.address))
print(student.model_dump())         # nesting survives the round trip

### Locking the model down

In [ ]:
from pydantic import ConfigDict


class StrictStudent(BaseModel):
    model_config = ConfigDict(
        extra="forbid",             # reject fields you did not ask for
        str_strip_whitespace=True,  # trim every string automatically
    )

    name: str


print(StrictStudent(name="  Ada  "))          # name='Ada' - trimmed

try:
    StrictStudent(name="Ada", is_admin=True)  # a field nobody declared
except ValidationError as error:
    print("extra ->", error.errors()[0]["msg"])

> Without `extra="forbid"`, `is_admin` would be **silently ignored** — and a typo in a field name
> would silently do nothing at all. That is a genuinely nasty class of bug.

---

## 17. Exercises

Fill in the blanks (`___`). Hints are in the comments.

### Q1: A dict, safely

Print the student's `cgpa`, falling back to `"not recorded"` when the key is missing.

In [ ]:
record = {"name": "Meera", "branch": "ECE"}

# Hint: one of these two crashes on a missing key, the other does not.
print(record.___("cgpa", "___"))

### Q2: A function with a default

Write `discount(price, percent=10)` that returns the price after the discount.

In [ ]:
def discount(price: float, ___: int = ___) -> float:
    return price - (price * ___ / 100)


print(discount(1000))            # expect 900.0
print(discount(1000, 25))        # expect 750.0

### Q3: Type hints

Add hints: the function takes a list of strings and returns a single string.

In [ ]:
def join_names(names: ___[___]) -> ___:
    return ", ".join(names)


print(join_names(["Ada", "Raj"]))

### Q4: Refuse bad input

Make `set_marks` raise a `ValueError` when the mark is outside 0–100.

In [ ]:
def set_marks(mark: int) -> int:
    if mark < 0 or mark > ___:
        ___ ValueError("mark must be between 0 and 100")
    return mark


try:
    set_marks(150)
except ___ as error:
    print("refused:", error)

### Q5: A class with a dunder

Give `Book` a `__str__` so `print(book)` shows `Dune by Herbert`.

In [ ]:
class Book:
    def __init__(self, title: str, author: str):
        self.title = title
        self.___ = author

    def ___(self) -> str:
        return f"{self.title} by {self.___}"


print(Book("Dune", "Herbert"))

### Q6: Inheritance

Make `Teacher` inherit from `Person` and override `role()`.

In [ ]:
class Person:
    def __init__(self, name: str):
        self.name = name

    def role(self) -> str:
        return "person"


class Teacher(___):
    def role(self) -> str:
        return "___"


print(Teacher("Ada").name, "->", Teacher("Ada").role())

### Q7: A dataclass

Turn this into a dataclass with a default `credits` of 4.

In [ ]:
from dataclasses import dataclass


@___
class Course:
    code: ___
    title: str
    credits: int = ___


print(Course("CS101", "Databases"))
print(Course("CS101", "Databases") == Course("CS101", "Databases"))   # expect True

### Q8: A Pydantic model with rules

`age` must be at least 16, and `name` at least 2 characters.

In [ ]:
class Candidate(BaseModel):
    name: str = Field(min_length=___)
    age: int = Field(___=16)


print(Candidate(name="Ada", age=20))

try_it(Candidate, name="A", age=12)      # expect two problems

---

## Key Takeaways

1. **A dict is how data travels in a backend.** Every request and response is one underneath.

2. **Type hints are documentation to Python and validation to FastAPI.** Python ignores them;
   FastAPI enforces them. That is why one word — `int` — is worth writing.

3. **Check, then refuse clearly.** `raise ValueError(...)` in plain Python becomes
   `raise HTTPException(404)` in an API. Same instinct.

4. **JSON is a dict with different punctuation.** `True` → `true`, single quotes → double.

5. **`self` is just the object the method was called on.** `ada.label()` is
   `Student.label(ada)`.

6. **The four pillars, in one line each:** encapsulation keeps data with the rule that guards it ·
   inheritance reuses a parent · polymorphism keeps the calling line unchanged · abstraction hides
   the how.

7. **Ask "is-a or has-a?"** Most real modelling is has-a — and a has-a relationship is what a
   database will later call a *relationship*.

8. **`@dataclass` writes `__init__`, `__repr__` and `__eq__`** — and now you know what it wrote.

9. **A Pydantic model is a dataclass that enforces its own types**, and its errors are *data* you
   can act on. That is the engine FastAPI runs on.

### Quick Reference

| Idea | The one-liner |
|---|---|
| `dict.get(key, default)` | never crashes on a missing key |
| `list[int]`, `str \| None` | the hints you will use all week |
| `raise ValueError(...)` | refuse bad input on purpose |
| `json.dumps` / `json.loads` | dict → JSON text → dict |
| `self` | the object this method was called on |
| `super().__init__(...)` | run the parent's version first |
| `__str__` / `__repr__` / `__eq__` | what print, the debugger, and `==` show |
| `@property` | a method you use without `()` |
| `ABC` + `@abstractmethod` | a contract children must honour |
| `@dataclass` | the boilerplate, written for you |
| `field(default_factory=list)` | a fresh list per object |
| `BaseModel` | a class that validates itself |
| `Field(ge=, min_length=, pattern=)` | rules on one value |
| `field_validator` / `model_validator` | one field · the whole object |
| `.model_dump()` | model → plain dict |
| `extra="forbid"` | reject fields nobody declared |

### 🏠 Homework

1. **Model something you know.** Pick anything with fields and rules — a hostel room, a library
   loan, a bus booking — and write it three ways: a plain class, a `@dataclass`, and a `BaseModel`
   with at least two `Field()` rules. Which one would you want to receive from the internet?
2. **Break it on purpose.** For your `BaseModel`, write five inputs that *should* be rejected, and
   check that each one is. Print the `.errors()` and read what came back.
3. **Find a has-a.** Model one relationship in your idea using composition, not inheritance, and
   write one sentence saying why inheritance would have been wrong.

---

**Next in this module:** **FastAPI**, in VS Code. Everything from here needs a running server, so
it moves to the local project — see `backend-engineering/SETUP.md` and
`backend-engineering/code/01_foundations/`. The type hints from section 6 and the `BaseModel`s from
section 14 are what it is built on.